In [1]:
import os
import time
import datetime
import threading
import requests
from PIL import Image
from pyngrok import ngrok
from selenium import webdriver
from selenium.webdriver.common.by import By
from flask import Flask, request, send_from_directory
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
# ======== 自訂區 ========
LINE_TOKEN = "QNFu6ERH0NP7O4DLWNY9o+kEnJjUCiIXX9nFU+ttXExTO3oJOiTos0EmLRzeEZjjpjVgF4RvkhgJdCvLLbbAUN ScTMemFdruEq0zSALZD0YdRL562yOgyVZi5nWoiVmJRXjJ7WdNKH3C5pVS6/KlqAdB04t89/1O/w1cDnyilFU="
USER_ID = "Uada04739a4ce5517e2b41e8cbe109c05"
TEAMS_URL = "https://teams.microsoft.com/l/meetup-join/19%3ameeting_YzY1NjA5MjQtNzU1Mi00NGJlLTk1NDEtN2IzMjZhZGQyYTRi%40thread.v2/0?context=%7b%22Tid%22%3a%22ca990a2d-cc4b-49b3-9a3c-200bf590f680%22%2c%22Oid%22%3a%227bac8b78-32e9-4092-b579-2a81c2c198a2%22%7d"
GUEST_NAME = "洪珮芠"
TARGET_TIME = "18:10"  # 等待到這時間執行
WAIT_BEFORE_JOIN = 5  # 頁面載入等待秒數

In [3]:
# ======== 啟動 Flask + ngrok 伺服器 ========
app = Flask(__name__)
lock = threading.Lock()

@app.route("/img/<path:filename>")
def get_image(filename):
    filepath = os.path.join(".", filename)
    if os.path.exists(filepath):
        from flask import send_file
        return send_file(filepath, mimetype="image/jpeg", cache_timeout=0)
    else:
        return "Not Found", 404

def start_ngrok():
    ngrok.kill()
    public_url = ngrok.connect(5000).public_url
    print("🌍 ngrok 公開網址：", public_url)
    print(f"➡️ 請在 LINE Developers 填入 Webhook URL：{public_url}/linebot")
    return public_url


def run_flask():
    app.run(port=5000, debug=False, use_reloader=False)


public_url = start_ngrok()
threading.Thread(target=run_flask, daemon=True).start()
# -------------------------------------------------------------------
# ==============================================

🌍 ngrok 公開網址： https://cc3dc10fd0c8.ngrok-free.app
➡️ 請在 LINE Developers 填入 Webhook URL：https://cc3dc10fd0c8.ngrok-free.app/linebot


 * Serving Flask app '__main__'
 * Debug mode: off


In [4]:
# 傳送文字訊息
def send_line_message(text):
    headers = {
        "Authorization": f"Bearer {LINE_TOKEN}",
        "Content-Type": "application/json"
    }
    body = {"to": USER_ID, "messages": [{"type": "text", "text": text}]}
    requests.post("https://api.line.me/v2/bot/message/push",
                  headers=headers, json=body)

 * Running on http://127.0.0.1:5000


Press CTRL+C to quit


In [ ]:
# 接收 LINE 使用者訊息
@app.route("/linebot", methods=["POST"])
def linebot():
    global TEAMS_URL, TARGET_TIME
    try:
        data = request.get_json()
        print("📥 收到 LINE webhook：", data)

        if not data or "events" not in data:
            return {"error": "Invalid payload"}, 400

        for event in data["events"]:
            if event["type"] == "message" and "text" in event["message"]:
                text = event["message"]["text"].strip()

                # 1️⃣ 判斷訊息是否為 Teams 連結
                if text.startswith("https://teams.microsoft.com/"):
                    with lock:
                        TEAMS_URL = text
                    send_line_message(f"🔗 已更新會議連結！")
                    print("✔️ 已更新 TEAMS_URL =", TEAMS_URL)
                    continue

                # 2️⃣ 判斷訊息是否為時間（格式 HH:MM）
                import re
                if re.match(r"^(?:[01]\d|2[0-3]):[0-5]\d$", text):
                    with lock:
                        TARGET_TIME = text
                    send_line_message(f"⏰ 已更新目標時間！")
                    print("✔️ 已更新 TARGET_TIME =", TARGET_TIME)
                    continue

                # 3️⃣ 其他文字
                send_line_message("❓ 請提供正確的 Teams 連結或時間（格式：HH:MM）。")

        return {"status": "ok"}, 200

    except Exception as e:
        print("❌ webhook error:", e)
        return {"error": str(e)}, 500

In [6]:
# 等待被允許進入會議
def wait_for_meeting_entry(driver, max_wait_minutes=10):
    print("⌛ 等待主持人允許中...".format(max_wait_minutes))
    start = time.time()

    while True:
        # 若 URL 改變或出現「離開」按鈕 → 成功進入
        if "meetingStage" in driver.current_url or driver.find_elements(
            By.XPATH, "//button[contains(., '離開') or contains(., 'Leave')]"
        ):
            print("✅ 已進入會議！")
            return True
        # 若畫面包含「有人會讓你進入」等提示 → 持續等待
        if driver.find_elements(
            By.XPATH,
            "//*[contains(text(), '讓你進入') or contains(text(), 'Someone in the meeting') or contains(text(), 'We’ll let you in')]",
        ):
            time.sleep(5)
        # 超時判斷
        if (time.time() - start) > max_wait_minutes * 60:
            print("⚠️ 等待超時。")
            return False
        time.sleep(5)

In [7]:
# 等待到指定時間
def wait_until_target():
    print(f"⏰ 等待直到 {TARGET_TIME} 執行自動加入會議...")
    while True:
        now = datetime.datetime.now().strftime("%H:%M")
        if now >= TARGET_TIME:
            send_line_message(f"🕓 時間到 {now}，開始自動加入 Teams 會議。")
            auto_join_meeting()
            break
        time.sleep(20)

In [8]:
# 自動加入 Teams 會議
def auto_join_meeting():
    global TEAMS_URL
    try:
        options = Options()
        options.add_argument("--disable-popup-blocking")
        options.add_argument("--no-default-browser-check")
        options.add_argument("--disable-infobars")
        options.add_argument("--disable-notifications")
        options.add_argument("--use-fake-ui-for-media-stream")

        driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=options
        )
        wait = WebDriverWait(driver, 30)

        driver.get("about:blank")
        driver.get(TEAMS_URL)
        print("✅ 開啟 Teams 網址中...")

        # 在瀏覽器中繼續
        try:
            btn = wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH, '//button[@aria-label="從這個瀏覽器加入會議"]')
                )
            )
            btn.click()
        except Exception as e:
            print("⚠️ 找不到『從這個瀏覽器加入會議』按鈕。", e)
        try:
            btn = wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH, '//button[@data-focus-target="gum-continue"]')
                )
            )
            btn.click()
        except:
            print("⚠️ 找不到『在無音訊或視訊的情況下繼續』按鈕。")
        time.sleep(WAIT_BEFORE_JOIN)

        # 輸入暱稱
        try:
            name_input = wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH, '//input[@data-tid="prejoin-display-name-input"]')
                )
            )
            name_input.clear()
            name_input.send_keys(GUEST_NAME)
        except:
            print("⚠️ 找不到『輸入您的名稱』輸入框。")
        time.sleep(1)

        # 關閉鏡頭、麥克風
        try:
            no_audio_btn = driver.find_element(
                By.XPATH, '//input[@type="radio" and @value="3"]'
            )
            no_audio_btn.click()
        except:
            print("⚠️ 找不到『不使用音訊』按鈕。")
        time.sleep(1)

        # 點「立即加入」
        try:
            join = driver.find_element(By.XPATH, '//button[@aria-label="立即加入"]')
            join.click()
        except:
            print("⚠️ 找不到「立即加入」按鈕。")
        
        # 等待被允許
        entered = wait_for_meeting_entry(driver, 15)

        # 回報 LINE
        if entered:
            send_line_message("✅ 已進入會議！")
        else:
            send_line_message("⚠️ 仍在等候主持人允許進入。")
    except Exception as e:
        send_line_message(f"❌ 執行錯誤：{e}")

In [ ]:
threading.Thread(target=wait_until_target, daemon=True).start()
send_line_message("✅ 系統啟動完成，等待時間到自動加入會議。")

⏰ 等待直到 18:10 執行自動加入會議...


✅ 開啟 Teams 網址中...


t=2025-11-14T18:23:08+0800 lvl=warn msg="failed to check for update" obj=updater err="Post \"https://update.equinox.io/check\": context deadline exceeded"


⌛ 等待主持人允許中...
✅ 已進入會議！


: 